# 2 · Find and validate

For one year (built with notebook 1): apply the current rules, rank the papers by similarity to
the confirmed facility papers, and create a **Google Sheet** to validate:

* **Validate**: candidates with the reason, evidence, corresponding author and e-mail. Fill
  `verdict` (yes / likely / no) and `note`.
* **Facility papers**: every confirmed paper of the year plus those you mark yes/likely, with
  corresponding author, e-mail, and whether the facility and the grant are acknowledged: use it
  for mail merge (e.g. Gmail's mail merge in Google Sheets, or the YAMM add-on).
* **Search misses**: confirmed papers the search did not flag, and why.

After validating, run section 4: it refreshes the Sheet, prints the confirmed DOIs to paste into
the repository's inbox, and re-ranks the other years with what you just confirmed.

Takes a few minutes. The Sheet and the e-mail addresses stay in your Drive.

## Settings

In [ ]:
FACILITY = 'aic-turku'
YEAR = 2024                               # the year to validate (built with notebook 1)
TOP_N = 100                               # check-list papers to include ("reads like facility papers")
DATA = '/content/drive/MyDrive/AIC-publications'
REPO = 'https://github.com/AIC-Turku/facility-publication-finder'
SHEETS_FOLDER_ID = ''                     # Drive folder id for the Sheets (default: the DATA folder)

## 1. Setup

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DATA, exist_ok=True)

# The code is cloned fresh into the temporary Colab disk every session; all data stays in DATA
# (Google Drive), so updating the code never touches results.
CODE = '/content/code'                    # temporary disk; rules and confirmed papers come with it
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '--ff-only', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, CODE], check=True)
%pip install -q "{CODE}[embed,openiris]"
os.environ['PUBS_DATA'] = DATA                                  # every command reads and writes data here
os.environ['PUBS_FACILITY'] = f'{CODE}/facilities/{FACILITY}'   # rules + confirmed papers
!aic-pubs check-facility

from google.colab import auth
auth.authenticate_user()                  # access to your Drive and Sheets, to create and read the Sheet
import google.auth, gspread
gc = gspread.authorize(google.auth.default()[0])
from aic_pubs import workflow

## 2. Search the year with the current rules

In [ ]:
if YEAR not in workflow.swept_years():
    raise SystemExit(f'{YEAR} is not built yet: run notebook 1 for it first.')
!aic-pubs rescreen --year {YEAR}
!aic-pubs validate --years {YEAR}

## 3. Rank and create the Sheet

The ranking learns from the confirmed papers in the repository plus everything marked **yes**
in the Sheets of other years, never from this year's own papers.

In [ ]:
url = workflow.prepare_sheet(gc, YEAR, top_n=TOP_N, folder_id=SHEETS_FOLDER_ID or None)
print('\nValidation Sheet:', url)

## 4. After validating: collect, file and learn

Validate in the Sheet's **Validate** tab (`verdict` dropdown):

* **yes**: the paper used the facility. It is filed (below) and joins *Facility papers*.
* **likely**: probably, but not certain. It is *not* filed; it appears in *Facility papers* so you
  can e-mail the authors, and you change it to **yes** when they confirm.
* **no**: not facility use. Nothing to do: the notes help improve the rules.
* *Search misses* needs nothing from staff: it is for whoever maintains the rules.

Then run the cells below (also days later: run *Settings* and *1. Setup* first). They refresh
the Sheet and print the new confirmed DOIs: **paste them into the repository's inbox** (link
printed) and commit; a GitHub Action files them under `papers/<year>.yaml`. The last cell
re-ranks the other years' Sheets with what you confirmed.

**Mail merge** (*Facility papers* tab): one address per row in `email`, further corresponding
authors in `other_contacts`. Gmail's built-in mail merge needs a Google Workspace account;
add-ons such as YAMM are third-party processors (check with your data protection officer).
The addresses were not given by the authors: the first message should say where the address
comes from, why it is used, and how to opt out; keep a do-not-contact list and delete the
contact columns after the campaign.

In [ ]:
url, inbox = workflow.collect(gc, YEAR, top_n=TOP_N)
print('Sheet:', url)
repo_web = REPO.removesuffix('.git')
print(f'\nPaste into {repo_web}/edit/main/facilities/{FACILITY}/inbox.txt :\n')
print(inbox or '(no new "yes" verdicts)')

In [ ]:
LEARN = True                              # re-rank the other years' Sheets with the new verdicts
if LEARN:
    for y, sheet_url in workflow.learn(gc, YEAR, top_n=TOP_N).items():
        print(y, sheet_url)